# E03: 
use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

# Solution:

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [3]:
words = open("names.txt", 'r').read().splitlines()

In [4]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [5]:
len(words)

32033

In [6]:
min(words, key=len), len(min(words, key=len))

('an', 2)

In [7]:
max(words, key=len), len(max(words, key=len))

('muhammadibrahim', 15)

In [8]:
# Get list of all the characters
chars = ['.'] + sorted(list(set(''.join(words)))) # ., a, b, c, d, .... x, y, z
len(chars)

27

In [9]:
# Create dictionary for mapping single character to its ID
stoi = {s:i for i, s in enumerate(chars)}

# Create another dictionary for reverse mapping
itos = {i:s for s, i in stoi.items()}

# Check if its correct
stoi['.'], stoi['a'], itos[0]

(0, 1, '.')

In [10]:
# Get all 27*27 combinations (26 alphabets and one '.')
all_combinations = [a + b for a in chars for b in chars]

# Create dictionary for mapping combination to its ID
ctoi={c:i for i,c in enumerate(all_combinations)}

# Create another dictionary for reverse mapping
itoc={i:c for c, i in ctoi.items()}

# Check if its correct
len(ctoi), ctoi['..'], itoc[1]

(729, 0, '.a')

**Note:** A universal dictionary (`combinations + chars`) would not work in this case because the character IDs would fall in the range **729–755**, while the output classes are expected to be in the range **0–26**.

Doing it the other way around (`chars + combinations`) would not work either, because the combination IDs would fall in the range **27–755**, while `one_hot` for the input expects IDs in the range **0–728** when `num_classes=729`.

Therefore, separate dictionaries are needed:
- `ctoi` → character → IDs `0–26`
- `stoi` → combination → IDs `0–728`


## Create train, dev and test split

In [11]:
g1 = torch.Generator().manual_seed(42)

perm = torch.randperm(len(words), generator=g1)
perm, perm.shape

(tensor([ 4348, 12372,  7029,  ...,  7956,  2399,  8375]), torch.Size([32033]))

In [12]:
words_shuffled = [words[i] for i in perm]

train_words = words_shuffled[:25626]
dev_words   = words_shuffled[25626:28830]
test_words  = words_shuffled[28830:32033]

## Trigram

In [16]:
# Creating the dataset
def create_dataset_trigram(words):
    xs, ys = [], []
    for w in words:
        chs = ['.','.'] + list(w) + ['.']  # eg ['.', '.', 'e', 'm', 'm', 'a', '.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ch12 = ch1+ch2
            idx12 = ctoi[ch12]
            idx3 = stoi[ch3]
            xs.append(idx12)
            ys.append(idx3)
    
    # Converting dataset into tensor
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    num = xs.numel()
    return xs, ys, num 

In [17]:
xs_train_t, ys_train_t, training_size_t = create_dataset_trigram(train_words)
xs_dev_t, ys_dev_t, dev_size_t = create_dataset_trigram(dev_words)
xs_test_t, ys_test_t, test_size_t = create_dataset_trigram(test_words)

In [18]:
print(xs_dev_t.shape)
print(ys_dev_t.shape)
print(dev_size_t)

torch.Size([22768])
torch.Size([22768])
22768


In [19]:
# Testing the created dataset for correctness
def unit_test_dataset(xs, ys):
    for i in range(10):
        if itos[ys[i].item()]=='.':
            print(itoc[xs[i].item()], itos[ys[i].item()], sep="")
            break
        print(itoc[xs[i].item()], itos[ys[i].item()], sep="")

## Trainset test
print("-------------First few samples in trainset--------------")
unit_test_dataset(xs_train_t, ys_train_t)
print(f"Trainset size: {training_size_t}")
print("---------------------------------------------------")

## Devset test
print("-------------First few samples in devset----------------")
unit_test_dataset(xs_dev_t, ys_dev_t)
print(f"Devset size: {dev_size_t}")
print("---------------------------------------------------")

## Testset test
print("-------------First few samples in testset----------------")
unit_test_dataset(xs_test_t, ys_test_t)
print(f"Testset size: {test_size_t}")
print("---------------------------------------------------")

-------------First few samples in trainset--------------
..e
.ed
edi
dis
iso
son
on.
Trainset size: 182819
---------------------------------------------------
-------------First few samples in devset----------------
..x
.xa
xav
avi
vie
ier
era
ra.
Devset size: 22768
---------------------------------------------------
-------------First few samples in testset----------------
..j
.jo
jos
ose
set
ett
tte
te.
Testset size: 22559
---------------------------------------------------


In [50]:
# CUDA 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Move data and weights to device
xs_train_t = xs_train_t.to(device)
ys_train_t = ys_train_t.to(device)
xs_dev_t = xs_dev_t.to(device)
ys_dev_t = ys_dev_t.to(device)
xs_test_t = xs_test_t.to(device)
ys_test_t = ys_test_t.to(device)
W_trigram = W_trigram.to(device)

Using: cuda


In [27]:
# Setting up the generator and initializing weights with random values
g = torch.Generator(device=device).manual_seed(312312)
W_trigram = torch.randn((729, 27), generator=g, device=device, requires_grad=True)

### Finding the best lambda
Tuning the strength of smoothing to find the best value of regularization parameter.

In [ ]:
# Candidates
lambdas = [
    0.000001,
    0.00001,
    0.0001,
    0.001,
    0.005,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0,
    5.0,
    10.0
]
losses = []
min_loss = float('inf')
optimal_lambda = 0

print("----------------Losses reported are losses at last iteration of training---------------")

# Training
for lambdaa in lambdas:
    for i in range(100):
        #----------Forward pass------------
        xenc = F.one_hot(xs_train_t, num_classes=729).float() #729
        logits = xenc @ W_trigram # (182819, 729) @ (729, 27) = (182819,27)
    
        counts = logits.exp() 
        probs = counts/counts.sum(1, keepdim=True) 
        loss = -probs[torch.arange(training_size_t, device=device), ys_train_t].log().mean()+lambdaa/(2*training_size_t)*(W_trigram**2).sum()

        train_loss = loss.item()
        
        #-----------Backward pass--------------
        W_trigram.grad = None 
        loss.backward()
    
        #------------Update weights--------------
        W_trigram.data += -70 * W_trigram.grad


        xenc = F.one_hot(xs_dev_t, num_classes=729).float()
        logits = xenc @ optimal_W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        logprobs = torch.log(probs)
        nlls = -logprobs[torch.arange(len(ys_dev_t)), ys_dev_t]
        
        dev_loss = nlls.mean().item()

    print(f"Train loss: {train_loss}, Dev loss: {dev_loss}, lambda used: {lambdaa}")
    
    # Keep track of (train loss, dev loss, lambda) and save best lambda and its losses along with weights
    losses.append((train_loss, dev_loss, lambdaa))
    if dev_loss < min_loss:
        min_loss = dev_loss
        optimal_lambda = lambdaa
        optimal_W = W_trigram
print(f"Min dev loss reported is: {min_loss} at lambda: {optimal_lambda}")

----------------Losses reported are losses at last iteration of training---------------
Train loss: 2.2375590801239014, Dev loss: 2.266716718673706, lambda used: 1e-06
Train loss: 2.2268741130828857, Dev loss: 2.2577874660491943, lambda used: 1e-05
Train loss: 2.220435380935669, Dev loss: 2.252664804458618, lambda used: 0.0001
Train loss: 2.2160041332244873, Dev loss: 2.249255657196045, lambda used: 0.001
Train loss: 2.2128493785858154, Dev loss: 2.246798276901245, lambda used: 0.005
Train loss: 2.2104849815368652, Dev loss: 2.244932174682617, lambda used: 0.01
Train loss: 2.210491895675659, Dev loss: 2.243507146835327, lambda used: 0.05
Train loss: 2.211616277694702, Dev loss: 2.242401361465454, lambda used: 0.1
Train loss: 2.2324037551879883, Dev loss: 2.241952419281006, lambda used: 0.5
Train loss: 2.2584431171417236, Dev loss: 2.242300271987915, lambda used: 1.0
Train loss: 2.418002128601074, Dev loss: 2.253302574157715, lambda used: 5.0
Train loss: 2.503664255142212, Dev loss: 2.2

In [61]:
losses

[(2.2375590801239014, 2.266716718673706, 1e-06),
 (2.2268741130828857, 2.2577874660491943, 1e-05),
 (2.220435380935669, 2.252664804458618, 0.0001),
 (2.2160041332244873, 2.249255657196045, 0.001),
 (2.2128493785858154, 2.246798276901245, 0.005),
 (2.2104849815368652, 2.244932174682617, 0.01),
 (2.210491895675659, 2.243507146835327, 0.05),
 (2.211616277694702, 2.242401361465454, 0.1),
 (2.2324037551879883, 2.241952419281006, 0.5),
 (2.2584431171417236, 2.242300271987915, 1.0),
 (2.418002128601074, 2.253302574157715, 5.0),
 (2.503664255142212, 2.2894091606140137, 10.0)]

In [48]:
# Generating names
for i in range(30):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        xenc = F.one_hot(torch.tensor([ix]).to(device), num_classes=729).float() 
        logits = xenc @ optimal_W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

nlene
brini
ory
eliah
tramyreon
iwvfjrzwwofxqan
jah
rukwzaegipewrenyxwoyismafigndeueelishmlkyvuha
bla
yer
moe
aishaivcbfgvxfukfdfpeya
penuri
anniynhadquine
yod
nen
cacwbivyprvbarr
den
ma
madroo
quanubasi
tavb
seden
zarithxstiela
tayah
sanuettzjror
mai
bagypndpine
thon
roselyn


## Evaluating Trigram

In [ ]:
# Evaluating on test_set 
xenc = F.one_hot(xs_test_t, num_classes=729).float()
logits = xenc @ optimal_W
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_t)), ys_test_t]

avg_nll_trigram_test = nlls.mean().item()


print(f'Test loss = {avg_nll_trigram_test}')


Test loss = 2.287733554840088


## Summary
When lambda is too small or too large, both the train and dev losses are higher. With very large lambda, the losses become substantially higher compared with very small lambda because the strong regularization constrains the weights too much. The train and dev losses are lowest at an intermediate lambda, showing that an appropriate amount of regularization gives the best performance.

Using the best lambda selected based on the dev loss, the final test loss is 2.287733554840088.